In [4]:
!pip install xgboost

   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/69.5 MB ? eta -:--:--
    --------------------------------------- 1.0/69.5 MB 2.9 MB/s eta 0:00:24
    --------------------------------------- 1.3/69.5 MB 3.4 MB/s eta 0:00:21
    --------------------------------------- 1.3/69.5 MB 3.4 MB/s eta 0:00:21
   - -------------------------------------- 2.4/69.5 MB 2.6 MB/s eta 0:00:27
   - -------------------------------------- 3.1/69.5 MB 2.6 MB/s eta 0:00:26
   - -------------------------------------- 3.1/69.5 MB 2.6 MB/s eta 0:00:26
   - -------------------------------------- 3.4/69.5 MB 2.3 MB/s eta 0:00:29
   -- ------------------------------------- 3.7/69.5 MB 2.1 MB/s eta 0:00:32
   -- ------------------------------------- 3.9/69.5 MB 1.9 MB/s eta 0:00:35
   -- ---------------------


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Ali Raza\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


^C


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import joblib

In [2]:
df=pd.read_csv("booking.csv")
print(df.shape)


(36285, 17)


In [3]:
df

,Booking_ID,number of adults,number of children,number of weekend nights,number of week nights,type of meal,car parking space,room type,lead time,market segment type,repeated,P-C,P-not-C,average price,special requests,date of reservation,booking status
0,INN00001,1,1,2,5,Meal Plan 1,0,Room_Type 1,224,Offline,0,0,0,88.00,0,10/2/2015,Not_Canceled
1,INN00002,1,0,1,3,Not Selected,0,Room_Type 1,5,Online,0,0,0,106.68,1,11/6/2018,Not_Canceled
2,INN00003,2,1,1,3,Meal Plan 1,0,Room_Type 1,1,Online,0,0,0,50.00,0,2/28/2018,Canceled
3,INN00004,1,0,0,2,Meal Plan 1,0,Room_Type 1,211,Online,0,0,0,100.00,1,5/20/2017,Canceled
4,INN00005,1,0,1,2,Not Selected,0,Room_Type 1,48,Online,0,0,0,77.00,0,4/11/2018,Canceled
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36280,INN36282,2,0,0,2,Meal Plan 2,0,Room_Type 1,346,Online,0,0,0,115.00,1,9/13/2018,Canceled
36281,INN36283,2,0,1,3,Meal Plan 1,0,Room_Type 1,34,Online,0,0,0,107.55,1,10/15/2017,Not_Canceled
36282,INN36284,2,0,1,3,Meal Plan 1,0,Room_Type 4,83,Online,0,0,0,105.61,1,12/26/2018,Not_Canceled
36283,INN36285,3,0,0,4,Meal Plan 1,0,Room_Type 1,121,Offline,0,0,0,96.90,1,7/6/2018,Not_Canceled


In [4]:
print(df.columns)
print(df.info())

Index(['Booking_ID', 'number of adults', 'number of children',
       'number of weekend nights', 'number of week nights', 'type of meal',
       'car parking space', 'room type', 'lead time', 'market segment type',
       'repeated', 'P-C', 'P-not-C', 'average price', 'special requests',
       'date of reservation', 'booking status'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 36285 entries, 0 to 36284
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Booking_ID                36285 non-null  str    
 1   number of adults          36285 non-null  int64  
 2   number of children        36285 non-null  int64  
 3   number of weekend nights  36285 non-null  int64  
 4   number of week nights     36285 non-null  int64  
 5   type of meal              36285 non-null  str    
 6   car parking space         36285 non-null  int64  
 7   room type                 36285 non-null  st

In [5]:
df['total_guests'] = df['number of adults'] + df['number of children']

useless=['Booking_ID','number of adults','number of children','P-C','date of reservation']

df=df.drop(columns=useless)
print(df.shape)

(36285, 13)


In [6]:
 # String spaces clean karein
df['booking status'] = df['booking status'].astype(str).str.strip()

# Space aur Underscore dono variations ko map karein
df['booking status'] = df['booking status'].map({
    'Canceled': 1,
    'Not_Canceled': 0,
    'Not Canceled': 0,  # In case space ho underscore ki jagah
})

# Verify Karein
print(df['booking status'].value_counts())

booking status
0    24396
1    11889
Name: count, dtype: int64


In [7]:

X=df.drop(columns=['booking status'])
y=df['booking status']
print(X.shape)


(36285, 12)


In [8]:
y

0        0
1        0
2        1
3        1
4        1
        ..
36280    1
36281    0
36282    0
36283    0
36284    0
Name: booking status, Length: 36285, dtype: int64

In [9]:
X['type of meal']=X['type of meal'].map({
    'Not Selected':0,
    'Meal Plan 1':1,
    'Meal Plan 2':2,
    'Meal Plan 3':3
})

X['room type']=X['room type'].map({
    'Room_Type 1':0,
    'Room_Type 2':1,
    'Room_Type 3':2,
    'Room_Type 4':3,
    'Room_Type 5':4,
    'Room_Type 6':5,
    'Room_Type 7':6
})

X['market segment type']=X['market segment type'].map({
    'Online':0,
    'Aviation':1,
    'Offline':2,
    'Corporate':3,
    'Complementary':4
})
 


In [10]:
print(X.info())
X

<class 'pandas.DataFrame'>
RangeIndex: 36285 entries, 0 to 36284
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   number of weekend nights  36285 non-null  int64  
 1   number of week nights     36285 non-null  int64  
 2   type of meal              36285 non-null  int64  
 3   car parking space         36285 non-null  int64  
 4   room type                 36285 non-null  int64  
 5   lead time                 36285 non-null  int64  
 6   market segment type       36285 non-null  int64  
 7   repeated                  36285 non-null  int64  
 8   P-not-C                   36285 non-null  int64  
 9   average price             36285 non-null  float64
 10  special requests          36285 non-null  int64  
 11  total_guests              36285 non-null  int64  
dtypes: float64(1), int64(11)
memory usage: 3.3 MB
None


,number of weekend nights,number of week nights,type of meal,car parking space,room type,lead time,market segment type,repeated,P-not-C,average price,special requests,total_guests
0,2,5,1,0,0,224,2,0,0,88.00,0,2
1,1,3,0,0,0,5,0,0,0,106.68,1,1
2,1,3,1,0,0,1,0,0,0,50.00,0,3
3,0,2,1,0,0,211,0,0,0,100.00,1,1
4,1,2,0,0,0,48,0,0,0,77.00,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
36280,0,2,2,0,0,346,0,0,0,115.00,1,2
36281,1,3,1,0,0,34,0,0,0,107.55,1,2
36282,1,3,1,0,3,83,0,0,0,105.61,1,2
36283,0,4,1,0,0,121,2,0,0,96.90,1,3


In [11]:

# column names travel with the data. This lets XGBoost remember the exact
# feature names/order it was trained on, and lets us catch mismatches later
# instead of silently feeding values into the wrong columns.
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Scaler = StandardScaler()
x_train = pd.DataFrame(Scaler.fit_transform(x_train), columns=X.columns, index=x_train.index)
x_test = pd.DataFrame(Scaler.transform(x_test), columns=X.columns, index=x_test.index)


In [12]:

model=XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

In [13]:
model.fit(x_train,y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method

In [14]:
y_pred=model.predict(x_test)

print(accuracy_score(y_test,y_pred))


0.8763952046300124


In [15]:
#   joblib.dump(model, 'hotel_cancellation_model.pkl')
# but the model was TRAINED ON SCALED DATA (Scaler.fit_transform above).
# If you only save the model, the Streamlit app has no way to scale new
# input the same way -> every prediction in the app was wrong/meaningless.
#
# save a dictionary containing the model, the fitted scaler, and the
# exact list/order of feature columns used during training.
joblib.dump(
    {
        "model": model,
        "scaler": Scaler,
        "features": list(X.columns),  # exact column order the model expects
    },
    "hotel_cancellation_model.pkl",
)
print("Model + Scaler + Feature list saved as `hotel_cancellation_model.pkl`!")


Model + Scaler + Feature list saved as `hotel_cancellation_model.pkl`!


In [16]:
import pandas as pd

# Direct high-risk test instance (column names/order must match training X exactly)
sample_high_risk = pd.DataFrame([{
    "number of weekend nights": 2,
    "number of week nights": 4,
    "type of meal": 0,
    "car parking space": 0,
    "room type": 0,
    "lead time": 250,
    "market segment type": 4,
    "repeated": 0,
    "P-not-C": 0,
    "average price": 180.0,
    "special requests": 0,
    "total_guests": 2,
}])[X.columns]  # reorder to match training column order, just in case

#  the model was trained on SCALED data, so raw values must be
# scaled with the same fitted Scaler before predicting, or the probability
# is meaningless.
sample_scaled = pd.DataFrame(Scaler.transform(sample_high_risk), columns=X.columns)

print("Cancellation Probability:", model.predict_proba(sample_scaled)[0][1] * 100)


Cancellation Probability: 79.90146
